# Topic 20 — Imbalanced Datasets
### ⭐ High priority for your cyberbullying research. Theory → the problem → fixes → threshold tuning.

**Class imbalance**: one class (the **majority class**, e.g. "not bullying") vastly outnumbers
another (the **minority class**, e.g. "bullying"). Left unaddressed, models learn to just predict
the majority class most of the time — high accuracy, but useless for actually catching the
minority class you usually care about most.

In [ ]:
!pip install imbalanced-learn -q
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score
)
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

rng = np.random.default_rng(0)

## 1. See the problem directly

A severely imbalanced dataset: 95% majority class, 5% minority class.

In [ ]:
X, y = make_classification(
    n_samples=1000, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, weights=[0.95, 0.05], class_sep=1.0, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

print("class counts in train:", np.bincount(y_train))

plt.figure(figsize=(5, 4))
plt.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], alpha=0.3, label="majority (0)")
plt.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], alpha=0.8, label="minority (1)", color="red")
plt.legend()
plt.title("Severely imbalanced data")
plt.show()

In [ ]:
baseline = LogisticRegression().fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

print("accuracy:", (y_pred_baseline == y_test).mean())
print()
print(classification_report(y_test, y_pred_baseline, target_names=["not_bullying", "bullying"], zero_division=0))
# Accuracy looks great. But look at recall for class 1 -- the model may be missing most/all
# of the minority class while still scoring 90%+ accuracy overall.

## 2. Fix 1: Random oversampling

Duplicate existing minority-class samples until the classes are balanced. Simple, but can lead to
overfitting since it's just literal copies of the same points.

In [ ]:
ros = RandomOverSampler(random_state=42)
X_ros, y_ros = ros.fit_resample(X_train, y_train)   # fit/resample TRAIN only -- never touch test data

print("before:", np.bincount(y_train))
print("after random oversampling:", np.bincount(y_ros))

model_ros = LogisticRegression().fit(X_ros, y_ros)
y_pred_ros = model_ros.predict(X_test)
print()
print(classification_report(y_test, y_pred_ros, target_names=["not_bullying", "bullying"], zero_division=0))

## 3. Fix 2: Random undersampling

Remove majority-class samples until classes are balanced. Simple, but throws away potentially
useful data — risky with already-small datasets.

In [ ]:
rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X_train, y_train)

print("before:", np.bincount(y_train))
print("after random undersampling:", np.bincount(y_rus))

model_rus = LogisticRegression().fit(X_rus, y_rus)
y_pred_rus = model_rus.predict(X_test)
print()
print(classification_report(y_test, y_pred_rus, target_names=["not_bullying", "bullying"], zero_division=0))

## 4. Fix 3: SMOTE (Synthetic Minority Oversampling Technique)

Instead of duplicating existing minority points, SMOTE creates NEW synthetic minority points by
interpolating between a minority sample and one of its nearest minority-class neighbors (KNN idea
from Topic 10, applied here). Usually the best default oversampling choice.

In [ ]:
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)

print("before:", np.bincount(y_train))
print("after SMOTE:", np.bincount(y_smote))

plt.figure(figsize=(5, 4))
plt.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], alpha=0.2, label="majority (original)")
plt.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], alpha=0.9, label="minority (original)", color="red")
new_points_mask = np.arange(len(y_smote)) >= len(y_train)
plt.scatter(X_smote[new_points_mask, 0], X_smote[new_points_mask, 1],
            alpha=0.5, label="SMOTE synthetic points", color="orange", marker="x")
plt.legend()
plt.title("SMOTE generates synthetic minority points between real ones")
plt.show()

model_smote = LogisticRegression().fit(X_smote, y_smote)
y_pred_smote = model_smote.predict(X_test)
print()
print(classification_report(y_test, y_pred_smote, target_names=["not_bullying", "bullying"], zero_division=0))

## 5. Fix 4: Class weights (no resampling needed)

Instead of changing the DATA, tell the model to penalize mistakes on the minority class more
heavily during training. Often simpler and avoids the risk of overfitting on duplicated/synthetic
data — usually the first thing worth trying.

In [ ]:
model_weighted = LogisticRegression(class_weight="balanced")
model_weighted.fit(X_train, y_train)   # trains on the ORIGINAL imbalanced data
y_pred_weighted = model_weighted.predict(X_test)

print(classification_report(y_test, y_pred_weighted, target_names=["not_bullying", "bullying"], zero_division=0))

# class_weight="balanced" automatically sets weights inversely proportional to class frequency:
# weight_c = n_samples / (n_classes * count_of_class_c)
n = len(y_train)
counts = np.bincount(y_train)
manual_weights = n / (2 * counts)
print("\nequivalent manual weights:", {0: manual_weights[0], 1: manual_weights[1]})

## 6. Comparing all approaches side by side

In [ ]:
results = {
    "baseline (no fix)": y_pred_baseline,
    "random oversample": y_pred_ros,
    "random undersample": y_pred_rus,
    "SMOTE": y_pred_smote,
    "class_weight=balanced": y_pred_weighted,
}

print(f"{'approach':<25}{'precision':<12}{'recall':<12}{'f1':<12}")
for name, preds in results.items():
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)
    print(f"{name:<25}{p:<12.3f}{r:<12.3f}{f:<12.3f}")
# Notice: the baseline likely has the WORST recall (missing most bullying cases) despite fine-looking
# accuracy -- every other approach should trade some precision for meaningfully better recall.

## 7. Threshold tuning (revisited from Topic 9) as an additional/alternative fix

In [ ]:
y_proba = baseline.predict_proba(X_test)[:, 1]

for threshold in [0.5, 0.3, 0.15]:
    preds_t = (y_proba >= threshold).astype(int)
    p = precision_score(y_test, preds_t, zero_division=0)
    r = recall_score(y_test, preds_t, zero_division=0)
    print(f"threshold={threshold}: precision={p:.3f}, recall={r:.3f}")
# Lowering the threshold on the ORIGINAL imbalanced model can also substantially improve recall,
# without touching the training data or model at all -- often combined with the fixes above.

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Regenerate the dataset with weights=[0.99, 0.01] (even more extreme imbalance) and rerun
#    the baseline classification_report -- how much worse does minority recall get?
# 2. Try RandomForestClassifier(class_weight="balanced") instead of LogisticRegression --
#    does it change the precision/recall tradeoff?
# 3. Combine SMOTE with class_weight="balanced" (SMOTE first, then train a weighted model on the
#    resampled data) -- does that help or is it redundant here?
# 4. For your actual cyberbullying dataset: once you know its real class balance, decide which
#    combination (SMOTE / class_weight / threshold tuning) you'll try first, and why.

---
### Next up: **Topic 21 — NLP Fundamentals** (text preprocessing — your NLP path starts here).

Say "next" when you're ready.